# Sankey Diagram Builder - EHR Validated Drug Repurposing Candidates

## Overview
This notebook creates Sankey diagrams to visualize indirect drug repurposing pathways from EHR-validated candidates.

**Data Flow:** Drug → Gene Target → Risk Gene (all literature-validated and PPI-verified)

## Input Data
The notebook uses EHR-validated indirect drug candidates:
- `Results/HPV positive EHR drug candidates validated indirect.csv`
- `Results/HPV negative EHR drug candidates validated indirect.csv`

Each row represents a drug-gene-risk connection where:
- **gene_connected_to**: Intermediate gene that the drug targets (literature-validated)
- **gene_root_gene**: Mutated/risk gene connected to the disease (literature-validated)  
- Both connections have literature support (PMIDs provided)
- **All connections are already PPI-verified at confidence ≥700 from STRING database**

## Key Features
1. **Data Transformation**: Aggregates multiple drug-gene connections into per-drug summaries
2. **XGB Importance Ranking**: Drugs sorted by XGBoost feature importance from EHR model
3. **Gene Family Grouping**: Reduces visual complexity by grouping related genes (e.g., FGFRs, RPSs)

## Note
All gene target → risk gene connections in the input data are pre-validated through:
- Literature evidence (PMIDs)
- Protein-protein interactions (STRING database, confidence ≥700)

No additional PPI filtering is needed - all drugs will appear in the visualizations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import xml.etree.ElementTree as ET
from tqdm import tqdm

## Load Data

In [ ]:
# Load results - each row is drug → gene_target → risk_gene connection
hpv_positive_indirect_results_raw = pd.read_csv('Results/HPV positive EHR drug candidates validated indirect.csv')
hpv_negative_indirect_results_raw = pd.read_csv('Results/HPV negative EHR drug candidates validated indirect.csv')

### remove the row where drug is "ACETYSALICYLIC ACID" and gen_conneected to is "TP53" in hpv_negative_indirect_results_raw to avoid looping, and rename to asprin
hpv_negative_indirect_results_raw = hpv_negative_indirect_results_raw[~((hpv_negative_indirect_results_raw['ehr_drug'] == 'ACETYLSALICYLIC ACID') & (hpv_negative_indirect_results_raw['gene_connected_to'] == 'TP53'))].copy()
hpv_negative_indirect_results_raw['ehr_drug'] = hpv_negative_indirect_results_raw['ehr_drug'].str.replace('ACETYLSALICYLIC ACID', 'ASPIRIN')
hpv_negative_indirect_results_raw['gene_drug'] = hpv_negative_indirect_results_raw['gene_drug'].str.replace('ACETYLSALICYLIC ACID', 'ASPIRIN')

print(f"HPV+ raw connections: {len(hpv_positive_indirect_results_raw)} rows")
print(f"HPV- raw connections: {len(hpv_negative_indirect_results_raw)} rows")
print(f"\nHPV+ unique drugs: {hpv_positive_indirect_results_raw['ehr_drug'].nunique()}")
print(f"HPV- unique drugs: {hpv_negative_indirect_results_raw['ehr_drug'].nunique()}")

In [ ]:
hpv_negative_indirect_results_raw

In [ ]:
extracted_target_df= pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

#### DrugBank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

In [ ]:
Drug_bank

#### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

In [ ]:
protein_interaction[(protein_interaction['Translated_protein_1'] == 'ESR1') & (protein_interaction['Translated_protein_2'] == 'PIK3CA')]

## Helper Functions

In [ ]:
def build_ppi_lookup(protein_interaction_df, confidence_threshold=700):
    """
    Build a set of valid protein-protein interactions from STRING database.
    
    Parameters:
    -----------
    protein_interaction_df : DataFrame
        Protein interaction data with columns: Translated_protein_1, Translated_protein_2, combined_score
    confidence_threshold : int
        Minimum confidence score (default: 700)
    
    Returns:
    --------
    set : Set of (gene1, gene2) tuples for valid interactions (bidirectional)
    """
    # Filter by confidence threshold
    ppi_filtered = protein_interaction_df[
        protein_interaction_df['combined_score'] >= confidence_threshold
    ].copy()
    
    # Remove rows with empty translated proteins
    ppi_filtered = ppi_filtered[
        (ppi_filtered['Translated_protein_1'] != '') & 
        (ppi_filtered['Translated_protein_2'] != '')
    ].copy()
    
    # Create bidirectional set of interactions
    valid_ppi = set()
    for _, row in ppi_filtered.iterrows():
        gene1 = row['Translated_protein_1']
        gene2 = row['Translated_protein_2']
        valid_ppi.add((gene1, gene2))
        valid_ppi.add((gene2, gene1))  # Bidirectional
    
    print(f"Built PPI lookup with {len(valid_ppi)} bidirectional interactions")
    print(f"  (from {len(ppi_filtered)} protein pairs with confidence >= {confidence_threshold})")
    return valid_ppi

def group_gene_families(gene_targets):
    """
    Group related genes into families to reduce visual complexity.
    
    Parameters:
    -----------
    gene_targets : list
        List of gene symbols
    
    Returns:
    --------
    dict : Mapping from original gene to family name (or self if no family)
    """
    family_patterns = {
        'SSTR': r'^SSTR[1-5]$',
        'RPS': r'^RPS\d+[A-Z]*$',
        'RPL': r'^RPL\d+[A-Z]*$',
        'MAPK': r'^MAPK\d+$',
        'MAP2K': r'^MAP2K\d+$',
        'MAP3K': r'^MAP3K\d+$',
        'HLA-A': r'^HLA-A\d+$',
        'HLA-B': r'^HLA-B\d+$',
        'HLA-C': r'^HLA-C\d+$',
        'HIST1H1': r'^HIST1H1[A-Z]$',
        'KRT': r'^KRT\d+[A-Z]*$',
        'FGFR': r'^FGFR[1-4]$',
        'PSM': r'^PSM[A-Z]\d+[A-Z]*$',
        'PSMB': r'^PSMB\d+$',
        'COL': r'^COL\d+A\d+$',
        'CYP': r'^CYP\d+[A-Z]\d+$',
        'ATP': r'^ATP\d+[A-Z]\d*$',
        'SLC': r'^SLC\d+A\d+$',
    }
    
    gene_to_family = {}
    for gene in gene_targets:
        matched = False
        for family, pattern in family_patterns.items():
            if re.match(pattern, gene):
                gene_to_family[gene] = f"{family} family"
                matched = True
                break
        if not matched:
            gene_to_family[gene] = gene  # Keep original if no family match
    
    return gene_to_family

## Core Processing Function

In [ ]:
def prepare_sankey_data_simple(raw_df, top_n=50, group_families=True):
    """
    Simplified: Build Sankey data directly from raw connection file.
    
    Each row in raw_df is a drug → gene_target → risk_gene connection.
    
    Parameters:
    -----------
    raw_df : DataFrame
        Raw EHR results with columns:
        - ehr_drug, gene_connected_to, gene_root_gene, xgb_absolute_importance
    top_n : int
        Number of top drugs to include
    group_families : bool
        Whether to group gene families (default: True)
    
    Returns:
    --------
    dict with 'link_counts', 'drugs', 'gene_targets', 'risk_genes', 'family_to_genes'
    """
    # Filter to top drugs
    df_filtered = raw_df
    
    print(f"\n📊 Building Sankey from {len(df_filtered)} connections across {len(df_filtered['ehr_drug'].unique())} drugs")
    
    # Build links directly from raw data
    links = []
    for _, row in df_filtered.iterrows():
        drug = row['ehr_drug']
        gene_target = row['gene_connected_to']
        risk_gene = row['gene_root_gene']
        
        # Drug → Gene Target
        links.append({
            'source': drug,
            'target': gene_target,
            'link_type': 'drug_to_target',
            'value': 1
        })
        
        # Gene Target → Risk Gene
        links.append({
            'source': gene_target,
            'target': risk_gene,
            'link_type': 'target_to_risk',
            'value': 1
        })
    
    # Convert to DataFrame and remove duplicates
    link_counts = pd.DataFrame(links).drop_duplicates(subset=['source', 'target', 'link_type']).copy()
    
    # Apply gene family grouping if requested
    family_to_genes = {}
    if group_families:
        gene_targets_all = link_counts[link_counts['link_type'] == 'drug_to_target']['target'].unique()
        gene_family_map = group_gene_families(gene_targets_all)
        
        # Create reverse mapping
        for gene, family in gene_family_map.items():
            if family != gene:
                if family not in family_to_genes:
                    family_to_genes[family] = []
                family_to_genes[family].append(gene)
        
        # Apply mapping
        link_counts['target'] = link_counts.apply(
            lambda row: gene_family_map.get(row['target'], row['target']) 
            if row['link_type'] == 'drug_to_target' else row['target'], axis=1
        )
        link_counts['source'] = link_counts.apply(
            lambda row: gene_family_map.get(row['source'], row['source']) 
            if row['link_type'] == 'target_to_risk' else row['source'], axis=1
        )
        
        # Remove duplicates created by grouping
        link_counts = link_counts.drop_duplicates(subset=['source', 'target', 'link_type']).copy()
    
    # Extract node lists
    drugs = link_counts[link_counts['link_type'] == 'drug_to_target']['source'].unique().tolist()
    gene_targets = link_counts[link_counts['link_type'] == 'drug_to_target']['target'].unique().tolist()
    risk_genes = link_counts[link_counts['link_type'] == 'target_to_risk']['target'].unique().tolist()
    
    print(f"   Drugs: {len(drugs)}")
    print(f"   Gene Targets: {len(gene_targets)}")
    print(f"   Risk Genes: {len(risk_genes)}")
    print(f"   Total Links: {len(link_counts)}")
    
    return {
        'link_counts': link_counts,
        'drugs': drugs,
        'gene_targets': gene_targets,
        'risk_genes': risk_genes,
        'family_to_genes': family_to_genes
    }

## Visualization Functions

In [ ]:
def create_combined_sankey(data_dict, title="Drug Repurposing Pathways", 
                          height=650, width=1400, link_scale=0.03, node_font_size=14, node_thickness=3):
    """
    Create a combined Sankey diagram showing all drugs.

    Parameters added/adjusted:
    - `link_scale`: scales down link `value`s to reduce visual thickness (default 0.03)
    - `node_font_size`: font size for node labels (default 14)
    - `node_thickness`: node box thickness (default 3)
    """
    link_counts = data_dict['link_counts']
    drugs = set(data_dict['drugs'])
    gene_targets = set(data_dict['gene_targets'])

    # Filter link_counts to only include gene targets that are actually connected to drugs
    connected_targets = link_counts[link_counts['link_type'] == 'drug_to_target']['target'].unique()

    # Filter out target→risk links for targets that aren't connected to any drug
    link_counts = link_counts[
        ~((link_counts['link_type'] == 'target_to_risk') & 
          (~link_counts['source'].isin(connected_targets)))
    ].copy()

    # Identify gene families present
    gene_families = sorted([gt for gt in gene_targets if 'family' in gt])

    # Build node list and mapping
    all_nodes = sorted(list(set(link_counts['source'].unique()) | set(link_counts['target'].unique())))
    node_to_idx = {node: idx for idx, node in enumerate(all_nodes)}

    # Create link indices
    sources = [node_to_idx[row['source']] for _, row in link_counts.iterrows()]
    targets = [node_to_idx[row['target']] for _, row in link_counts.iterrows()]
    # scale down values to reduce thickness (keep a small floor to avoid zeros)
    raw_values = link_counts['value'].tolist()
    values = [max(v * link_scale, 0.001) for v in raw_values]

    # Assign colors
    node_colors = []
    node_x = []
    for node in all_nodes:
        if node in drugs:
            node_colors.append('rgba(31, 119, 180, 0.8)')  # Blue
            node_x.append(0.001)
        elif node in gene_targets:
            node_colors.append('rgba(44, 160, 44, 0.8)')   # Green
            node_x.append(0.5)
        else:
            node_colors.append('rgba(214, 39, 40, 0.8)')   # Red
            node_x.append(0.999)

    # Create figure with adjusted node font and more transparent link color (grey)
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=4,
            thickness=node_thickness,
            line=dict(color="black", width=0.5),
            label=all_nodes,
            color=node_colors,
            x=node_x,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color='rgba(128, 128, 128, 0.12)'
        ),
        textfont=dict(size=node_font_size, color='black')
    )])

    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        font=dict(size=node_font_size),
        height=height,
        width=width
    )

    # Add gene family legend if families are present
    if gene_families:
        family_to_genes = data_dict.get('family_to_genes', {})
        legend_lines = ["<b>Gene Families:</b>"]

        for family in gene_families:
            if family in family_to_genes:
                genes = family_to_genes[family]
                # Show first 5 genes, indicate if there are more
                if len(genes) <= 5:
                    gene_list = ", ".join(genes)
                else:
                    gene_list = ", ".join(genes[:5]) + f", ... (+{len(genes)-5})"
                legend_lines.append(f"• <b>{family}</b>: {gene_list}")
            else:
                legend_lines.append(f"• {family}")

        legend_text = "<br>".join(legend_lines)
        fig.add_annotation(
            text=legend_text,
            xref="paper", yref="paper",
            x=0.01, y=1.05,
            xanchor="left", yanchor="top",
            showarrow=False,
            font=dict(size=9, family="Arial"),
            align="left",
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="black",
            borderwidth=1,
            borderpad=8
        )

    return fig


def create_individual_sankeys(data_dict, drug_list=None, title_prefix="",
                              ncols=2, height=800, width=1200, 
                              risk_gene_threshold=0, link_scale=0.015, node_font_size=15, node_thickness=2):
    """
    Create a grid of individual Sankey diagrams, one per drug.

    Adjusted defaults to reduce visual size and link thickness while keeping fonts readable.
    """
    link_counts = data_dict['link_counts']
    gene_targets = set(data_dict['gene_targets'])

    # Get all gene families in the data
    all_gene_families = sorted([gt for gt in gene_targets if 'family' in gt])

    if drug_list is None:
        drug_list = data_dict['drugs'][:10]  # Default to top 10

    # Calculate grid dimensions
    n_drugs = len(drug_list)
    nrows = (n_drugs + ncols - 1) // ncols  # Ceiling division

    # Create subplots
    subplot_titles = [drug.title() for drug in drug_list]
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=subplot_titles,
        specs=[[{'type': 'sankey'}] * ncols for _ in range(nrows)],
        vertical_spacing=0.08,
        horizontal_spacing=0.03
    )

    # Process each drug
    for idx, drug in enumerate(drug_list):
        row = (idx // ncols) + 1
        col = (idx % ncols) + 1

        # Get links for this drug
        drug_to_target_links = link_counts[
            (link_counts['link_type'] == 'drug_to_target') & 
            (link_counts['source'] == drug)
        ]

        # Get the gene targets for this drug
        drug_gene_targets = drug_to_target_links['target'].unique()

        # Get target→risk links where the source is one of this drug's gene targets
        target_to_risk_links = link_counts[
            (link_counts['link_type'] == 'target_to_risk') &
            (link_counts['source'].isin(drug_gene_targets))
        ]

        # Combine both link types
        drug_links = pd.concat([drug_to_target_links, target_to_risk_links])

        if len(drug_links) == 0:
            print(f"   ⚠️  Skipping {drug}: No links found after getting drug_to_target and target_to_risk")
            continue

        # Debug counts
        drug_to_target_count = len(drug_links[drug_links['link_type'] == 'drug_to_target'])
        target_to_risk_count = len(drug_links[drug_links['link_type'] == 'target_to_risk'])

        if target_to_risk_count == 0 and drug_to_target_count > 0:
            print(f"   ⚠️  Skipping {drug}: Has {drug_to_target_count} gene targets but NO target→risk connections")
            continue

        # Filtering for large target lists remains unchanged...
        max_targets_threshold = 15
        if len(drug_gene_targets) > max_targets_threshold:
            target_to_risk = drug_links[drug_links['link_type'] == 'target_to_risk']
            risk_gene_counts = target_to_risk.groupby('target').size()
            risk_genes_keep = risk_gene_counts[risk_gene_counts >= risk_gene_threshold].index.tolist()

            drug_links = drug_links[
                (drug_links['link_type'] == 'drug_to_target') |
                ((drug_links['link_type'] == 'target_to_risk') & 
                 (drug_links['target'].isin(risk_genes_keep)))
            ].copy()

            if len(drug_links) == 0:
                continue

            drug_to_target_filtered = drug_links[drug_links['link_type'] == 'drug_to_target']
            target_to_risk_filtered = drug_links[drug_links['link_type'] == 'target_to_risk']
            gene_targets_with_connections = target_to_risk_filtered['source'].unique()

            drug_links = drug_links[
                (drug_links['link_type'] == 'target_to_risk') |
                ((drug_links['link_type'] == 'drug_to_target') & 
                 (drug_links['target'].isin(gene_targets_with_connections)))
            ].copy()

            if len(drug_links) == 0:
                continue

        # Build node list in correct order: drug first, then gene targets, then risk genes
        drug_to_target_final = drug_links[drug_links['link_type'] == 'drug_to_target']
        target_to_risk_final = drug_links[drug_links['link_type'] == 'target_to_risk']

        drug_nodes_layer = [drug]
        gene_target_nodes_layer = sorted(drug_to_target_final['target'].unique().tolist())
        risk_gene_nodes_layer = sorted(target_to_risk_final['target'].unique().tolist())

        drug_nodes = drug_nodes_layer + gene_target_nodes_layer + risk_gene_nodes_layer
        node_to_idx = {node: i for i, node in enumerate(drug_nodes)}

        # Create link indices and scaled values
        sources = [node_to_idx[row['source']] for _, row in drug_links.iterrows()]
        targets = [node_to_idx[row['target']] for _, row in drug_links.iterrows()]
        raw_vals = drug_links['value'].tolist() if 'value' in drug_links.columns else [1] * len(drug_links)
        values = [max(v * link_scale, 0.001) for v in raw_vals]

        # Assign colors and fonts
        node_colors = []
        for node in drug_nodes:
            if node == drug:
                node_colors.append('rgba(31, 119, 180, 0.9)')  # Blue
            elif node in gene_targets:
                node_colors.append('rgba(44, 160, 44, 0.9)')   # Green
            else:
                node_colors.append('rgba(214, 39, 40, 0.9)')   # Red

        # Add trace with reduced thickness and grey links
        fig.add_trace(
            go.Sankey(
                arrangement='snap',
                valueformat='.3f',
                node=dict(
                    pad=2,
                    thickness=node_thickness,
                    line=dict(color="black", width=0.5),
                    label=drug_nodes,
                    color=node_colors,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color='rgba(128, 128, 128, 0.12)'
                ),
                textfont=dict(size=node_font_size, color='black', family='Arial')
            ),
            row=row,
            col=col
        )

    # Update layout
    fig.update_layout(
        title={
            'text': f"{title_prefix}Individual Drug Pathways<br>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16}
        },
        height=height,
        width=width,
        showlegend=False,
        font=dict(size=node_font_size),
        margin=dict(t=80, b=20, l=20, r=20),
    ) 

    # Update subplot titles (drug names)
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=15, color='black', family='Arial')

    # Add gene family legend if families are present
    if all_gene_families:
        family_to_genes = data_dict.get('family_to_genes', {})
        legend_lines = ["<b>Gene Families:</b>"]

        for family in all_gene_families:
            if family in family_to_genes:
                genes = family_to_genes[family]
                if len(genes) <= 3:
                    gene_list = ", ".join(genes)
                else:
                    gene_list = ", ".join(genes[:3]) + f", ... (+{len(genes)-3})"
                legend_lines.append(f"• <b>{family}</b>: {gene_list}")
            else:
                legend_lines.append(f"• {family}")

        legend_text = "<br>".join(legend_lines)
        fig.add_annotation(
            text=legend_text,
            xref="paper", yref="paper",
            x=0.01, y=1,
            xanchor="left", yanchor="top",
            showarrow=False,
            font=dict(size=9, family="Arial"),
            align="left",
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="black",
            borderwidth=1,
            borderpad=6
        )

    return fig


## Wrapper Functions

In [ ]:
def create_sankey_diagrams(cohort='HPV+', top_n=10, 
                          show_combined=True, show_individual=True):
    """
    Simplified: Create all Sankey diagrams for a cohort directly from raw data.
    
    Parameters:
    -----------
    cohort : str
        'HPV+' or 'HPV-'
    top_n : int
        Number of top drugs to include
    show_combined : bool
        Whether to show combined multi-drug diagram
    show_individual : bool
        Whether to show individual per-drug diagrams
    
    Returns:
    --------
    dict with keys 'combined_fig' and 'individual_fig' (if requested)
    """
    # Select data
    if cohort.upper() == 'HPV+':
        raw_df = hpv_positive_indirect_results_raw
        title_prefix = "HPV+ "
    else:
        raw_df = hpv_negative_indirect_results_raw
        title_prefix = "HPV- "
    
    # Prepare data
    print(f"\n{'='*60}")
    print(f"Processing {cohort} cohort (Top {top_n})")
    print(f"{'='*60}")
    
    data_dict = prepare_sankey_data_simple(
        raw_df=raw_df,
        top_n=top_n,
        group_families=True
    )
    
    results = {}
    results['data'] = data_dict
    
    # Check if we have any data to plot
    if len(data_dict['drugs']) == 0:
        print(f"\n❌ No valid drugs found for {cohort}. Cannot create diagrams.")
        return results
    
    # Combined diagram
    if show_combined:
        print(f"\n🎨 Creating combined diagram...")
        results['combined_fig'] = create_combined_sankey(
            data_dict,
            title=f"{title_prefix}Top {top_n} Medications: Drug → Gene Target → Risk Gene"
        )
        results['combined_fig'].show()
    
    # Individual diagrams
    if show_individual:
        print(f"\n🎨 Creating individual drug diagrams...")
        drug_list = data_dict['drugs'][:top_n]
        
        results['individual_fig'] = create_individual_sankeys(
            data_dict,
            drug_list=drug_list,
            title_prefix=title_prefix
        )
        results['individual_fig'].show()
    
    return results

In [ ]:
hpv_positive_indirect_results_raw['Number of root_gene support'] = hpv_positive_indirect_results_raw['gene_root_literature_support'].apply(lambda x: len(x.split(',')) if pd.notnull(x) else 0)
hpv_positive_indirect_results_raw


In [ ]:
# HPV+ cohort: Create Sankey diagrams
results_hpv_pos = create_sankey_diagrams(
    cohort='HPV+',
    top_n=10,
    show_combined=True,
    show_individual=True
)

In [ ]:
hpv_negative_indirect_results_raw['Number of root_gene support'] = hpv_negative_indirect_results_raw['gene_root_literature_support'].apply(lambda x: len(x.split(',')) if pd.notnull(x) else 0)
hpv_negative_indirect_results_raw

In [ ]:
# HPV- cohort: Create Sankey diagrams
results_hpv_neg = create_sankey_diagrams(
    cohort='HPV-',
    top_n=10,
    show_combined=True,
    show_individual=True
)

In [ ]:
Drug_bank[Drug_bank['drug']=='Acetylsalicylic acid']

In [ ]:
hpv_negative_indirect_results_raw[hpv_negative_indirect_results_raw['ehr_drug'] == 'METHYLPREDNISOLONE']

In [ ]:
hpv_negative_indirect_results_raw[hpv_negative_indirect_results_raw['ehr_drug'] == 'LEVOTHYROXINE']

In [ ]:
hpv_negative_indirect_results_raw[hpv_negative_indirect_results_raw['ehr_drug'] == 'MELATONIN']

### Single drug double checks

In [ ]:
# Create diagram for a single specific drug (must be in top N)

def create_single_drug_diagram(drug_name, cohort='HPV+', top_n=100):
    """
    Print a concise summary for a specific drug and optionally create a Sankey
    panel for that drug using the raw connection data.

    Parameters
    ----------
    drug_name : str
        Drug name (case-insensitive)
    cohort : str
        'HPV+' or 'HPV-'
    top_n : int
        Consider top `top_n` drugs by `xgb_absolute_importance` when checking membership
    """
    # Select raw data for cohort
    if cohort.upper() == 'HPV+':
        raw_df = hpv_positive_indirect_results_raw
    else:
        raw_df = hpv_negative_indirect_results_raw

    # Ensure expected columns exist
    required = ['ehr_drug', 'gene_connected_to', 'gene_root_gene', 'xgb_absolute_importance']
    missing = [c for c in required if c not in raw_df.columns]
    if missing:
        print(f"Input data missing required columns: {missing}")
        return

    # Top drugs by importance
    drug_importance = raw_df.groupby('ehr_drug')['xgb_absolute_importance'].first().sort_values(ascending=False)
    top_drug_names = drug_importance.head(top_n).index.tolist()

    print(f"\n{'='*70}")
    print(f"🔍 Top {top_n} Drugs in {cohort}")
    print(f"{'='*70}\n")

    # Check whether the requested drug is present in the raw data
    drugs_available = raw_df['ehr_drug'].dropna().unique().tolist()
    match = None
    for d in drugs_available:
        if d.lower() == drug_name.lower():
            match = d
            break

    if match is None:
        print(f"❌ Drug '{drug_name}' not found in raw {cohort} data")
        print(f"Available drugs (sample): {', '.join(drug_importance.head(20).index.tolist())}...")
        return

    # Show basic stats for the drug
    drug_rows = raw_df[raw_df['ehr_drug'].str.lower() == match.lower()].copy()
    gene_targets = drug_rows['gene_connected_to'].dropna().unique().tolist()
    risk_genes = drug_rows['gene_root_gene'].dropna().unique().tolist()
    connections_count = len(drug_rows)
    importance = drug_rows['xgb_absolute_importance'].dropna().unique()
    importance_val = importance[0] if len(importance) > 0 else None

    print(f"✅ Drug: {match} (cohort={cohort})")
    print(f"   → XGB importance: {importance_val}")
    print(f"   → Gene targets: {len(gene_targets)}")
    if gene_targets:
        print(f"     • {', '.join(gene_targets[:10])}{'...' if len(gene_targets)>10 else ''}")
    print(f"   → Risk genes: {len(risk_genes)}")
    if risk_genes:
        print(f"     • {', '.join(risk_genes[:10])}{'...' if len(risk_genes)>10 else ''}")
    print(f"   → Total raw connections: {connections_count}\n")

    # Optionally, create and show a single-drug Sankey if the drug is within top_n
    if match in top_drug_names:
        print("Creating single-drug Sankey (using prepared data)...")
        try:
            # Prepare data_dict with a higher top_n to ensure inclusion
            data_dict = prepare_sankey_data_simple(raw_df, top_n=max(top_n, 200))
            fig = create_individual_sankeys(data_dict, drug_list=[match], ncols=1, height=800, width=1200, title_prefix=f"{cohort} ")
            fig.show()
        except Exception as e:
            print(f"Failed to create Sankey for {match}: {e}")
    else:
        print(f"Note: '{match}' is not in the top {top_n}. Use a larger `top_n` to visualize.")

    print(f"{'='*70}\n")


In [ ]:
# # Create diagram for a single specific drug
# create_single_drug_diagram('pomalidomide', cohort='HPV-')


In [ ]:
### examples of checking specific genes and drugs